In [1]:
from pathlib import Path
import pandas as pd

In [2]:
def find_project_root():

    current = Path.cwd().resolve()

    while current != current.parent:

        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

METRICS_DIR = (
    PROJECT_ROOT
    / "results"
    / "metrics"
)
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
model_a_df = pd.read_csv(
    METRICS_DIR
    / "model_a_metrics.csv"
)

model_b_df = pd.read_csv(
    METRICS_DIR
    / "model_b_metrics.csv"
)

custom_comparison = pd.concat(
    [
        model_a_df,
        model_b_df
    ],
    ignore_index=True
)

custom_comparison


,Model,Parameters,Estimated FP32 Size (KB),Actual Model Size (KB),Average Epoch Time (s),Inference Time (ms/image),Test Accuracy,Macro Precision,Macro Recall
0,Model A,45643,178.292969,182.848633,55.163382,0.359534,0.992993,0.990300,0.993890
1,Model B,25526,99.710938,105.735352,47.847918,0.341877,0.977227,0.980785,0.982066


In [4]:
model_b_df = pd.read_csv(
    METRICS_DIR
    / "model_b_metrics.csv"
)

mobile_df = pd.read_csv(
    METRICS_DIR
    / "mobilenetv2_metrics.csv"
)

efficient_df = pd.read_csv(
    METRICS_DIR
    / "efficientnetb0_metrics.csv"
)

model_comparison_df = pd.read_csv(
    TABLES_DIR / "model_a_vs_model_b.csv", index_col=0
)
model_b_macs = int(model_comparison_df.loc["Model B", "MACs per Image"])

In [5]:
model_b_final = model_b_df.copy()

model_b_final["Model Size (MB)"] = model_b_final["Actual Model Size (KB)"] / 1024
model_b_final["MACs per Image"] = model_b_macs

In [6]:
comparison_columns = [
    "Model",
    "Parameters",
    "MACs per Image",
    "Model Size (MB)",
    "Average Epoch Time (s)",
    "Inference Time (ms/image)",
    "Test Accuracy",
    "Macro Precision",
    "Macro Recall",
]

model_b_selected = model_b_final[comparison_columns]
mobile_selected = mobile_df[comparison_columns]
efficient_selected = efficient_df[comparison_columns]

In [7]:
final_comparison = pd.concat(
    [
        model_b_selected,
        mobile_selected,
        efficient_selected
    ],
    ignore_index=True
)

final_comparison

,Model,Parameters,MACs per Image,Model Size (MB),Average Epoch Time (s),Inference Time (ms/image),Test Accuracy,Macro Precision,Macro Recall
0,Model B,25526,1420800,0.103257,47.847918,0.341877,0.977227,0.980785,0.982066
1,MobileNetV2,2278955,24503552,8.929881,59.172997,0.598415,1.000000,1.000000,1.000000
2,EfficientNetB0,4062631,32021632,15.786809,67.178107,0.670844,0.998248,0.998728,0.999118


In [8]:
final_comparison.to_csv(
    PROJECT_ROOT
    / "results"
    / "tables"
    / "final_model_comparison.csv",
    index=False
)

## Task 6: final comparison and trade-offs

The saved test results show that Model B is the smallest option by a wide margin: it has 25,526 parameters, about 0.10 MB of weights, and 1.42 million MACs per 64 ? 64 image. Its test accuracy is 97.72%. MobileNetV2 reaches 100.00% accuracy with 2.28 million parameters, about 8.93 MB, and 24.50 million MACs. EfficientNet-B0 reaches 99.82% accuracy with 4.06 million parameters, about 15.79 MB, and 32.02 million MACs.

The pretrained networks improve test accuracy by about 2.28 and 2.10 percentage points over Model B, but require substantially more memory and computation. Model B also has the lowest recorded epoch and inference times in this comparison. It is therefore the strongest fit when the 100,000-parameter limit and edge-device memory budget dominate; a pretrained lightweight model is preferable when the extra accuracy is worth its larger footprint and compute cost.

These conclusions use the saved metrics produced with the shared 70/15/15 split. The report should include this table and discussion, along with the Task 4 custom-model comparison.